# Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 23.04.2024
- Date of Last Modification: 24.04.2024

## 1. Setup

### 1.1 Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
import pickle
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import scgpt as scg
from datasets import load_from_disk

from nichejepa import tokenizers
from nichejepa import utils

### 1.2 Run Notebook Setup

In [3]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### 1.3 Configure Paths and Create Directories

In [4]:
# Define paths
st_data_folder_path = "../datasets/st_data" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
cell_gene_mean_dictionary_file = "../datasets/cell_gene_mean_dictionary.pkl"
niche_gene_mean_dictionary_file = "../datasets/niche_gene_mean_dictionary.pkl"
geneformer_token_dictionary_file = "../datasets/geneformer_token_dictionary.pkl"
token_dictionary_file = "../datasets/token_dictionary.pkl"

## 2. Tokenization

### 2.1 Create Token Dictionary

In [ ]:
# Create token dict for nichejepa with cell and niche tokens
with open(geneformer_token_dictionary_file, "rb") as f:
    token_dict = pickle.load(f)

new_token_dict = {
    (key + "_cell" if "ENS" in key else key): value for key, value in token_dict.items()}
new_token_dict_copy = new_token_dict.copy()
i = 0
for key in new_token_dict_copy.keys():
    if "ENS" in key:
        new_token_dict[key.replace("cell", "niche")] = 25426 + i
        i += 1
del(new_token_dict_copy)

with open(nichejepa_token_dictionary_file, "wb") as file:
    pickle.dump(new_token_dict, file)

### 2.2 Compute Normalization Factors for Cell and Niche Genes

In [9]:
dataset = "nanostring_cosmx_human_brain"
tokenized_data_folder_path = f"{st_data_gold_folder_path}/{dataset}.dataset"

In [10]:
adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset}.h5ad")

In [8]:
# Normalize gene counts by read depth
norm_counts = adata.X / adata.X.sum(1).reshape(-1, 1) * 10000

# Compute non-zero means of read-depth normalized gene counts
masked_data = np.ma.masked_equal(norm_counts, 0)
nzmeans_cell = np.ma.mean(masked_data, axis=0)
nzmeans_cell = nzmeans_cell.filled(0)

# Normalize read-depth normalized gene counts by non-zero means
norm_counts = norm_counts / nzmeans_cell.reshape(1, -1)

In [9]:
# Normalize gene counts by read depth
norm_counts_niche = adata.layers["counts_niche"] / adata.layers["counts_niche"].sum(1).reshape(-1, 1) * 10000

# Compute non-zero means of read-depth normalized gene counts
masked_data_niche = np.ma.masked_equal(norm_counts_niche, 0)
nzmeans_niche = np.ma.mean(masked_data_niche, axis=0)
nzmeans_niche = nzmeans_niche.filled(0)

# Normalize read-depth normalized gene counts by non-zero means
norm_counts_niche = norm_counts_niche / nzmeans_niche.reshape(1, -1)

In [10]:
# Create nzmean dictionary for tokenizer
cell_gene_nzmean_dict = {}
niche_gene_nzmean_dict = {}

for gene, nzmean_cell, nzmean_niche in zip(adata.var["ensembl_id"].values, nzmeans_cell, nzmeans_niche):
    cell_gene_nzmean_dict[gene] = nzmean_cell
    niche_gene_nzmean_dict[gene] = nzmean_niche

In [21]:
# Store dictionary
with open(cell_gene_mean_dictionary_file, "wb") as file:
    pickle.dump(cell_gene_nzmean_dict, file)

with open(niche_gene_mean_dictionary_file, "wb") as file:
    pickle.dump(niche_gene_nzmean_dict, file)

### 2.3 Run Tokenizer

In [5]:
tk = tokenizers.STRankTokenizer(
    cell_gene_mean_file=cell_gene_mean_dictionary_file,
    niche_gene_mean_file=niche_gene_mean_dictionary_file,
    token_dictionary_file=token_dictionary_file)

In [14]:
tk.tokenize_data(input_directory=st_data_silver_folder_path,
                 output_directory=st_data_gold_folder_path,
                 output_file_prefix=dataset)

Tokenizing '../datasets/st_data/silver/nanostring_cosmx_human_brain.h5ad'...
'../datasets/st_data/silver/nanostring_cosmx_human_brain.h5ad' has no column 'filter_pass'; tokenizing all cells.
Creating Hugging Face dataset...


Saving the dataset (7/7 shards): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 188686/188686 [00:05<00:00, 36620.13 examples/s]


In [19]:
# Load the dataset
dataset = load_from_disk(tokenized_data_folder_path)

In [22]:
len(dataset["input_ids"][0])

2048

In [24]:
2+3

5

In [38]:
import json
import torch

from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.model import TransformerModel
from pathlib import Path

In [42]:
n_bins = 51
pad_value = -2
n_input_bins = n_bins

In [34]:
model_dir = Path("../scgpt/pretrained_model")
model_config_file = model_dir / "args.json"
model_file = model_dir / "best_model.pt"
vocab_file = model_dir / "vocab.json"

vocab = GeneVocab.from_file(vocab_file)
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

# Retrieve model parameters from config files
with open(model_config_file, "r") as f:
    model_configs = json.load(f)
print(
    f"Resume model from {model_file}, the model args will override the "
    f"config {model_config_file}."
)
embsize = model_configs["embsize"]
nhead = model_configs["nheads"]
d_hid = model_configs["d_hid"]
nlayers = model_configs["nlayers"]
n_layers_cls = model_configs["n_layers_cls"]

Resume model from ../scgpt/pretrained_model/best_model.pt, the model args will override the config ../scgpt/pretrained_model/args.json.


In [46]:
import torch

In [47]:
torch.__version__

'2.1.2+cu121'

In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    vocab=vocab,
    pad_value=pad_value,
    n_input_bins=n_input_bins,
)

try:
    model.load_state_dict(torch.load(model_file))
    print(f"Loading all model params from {model_file}")
except:
    # only load params that are in the model and match the size
    model_dict = model.state_dict()
    pretrained_dict = torch.load(model_file)
    pretrained_dict = {
        k: v
        for k, v in pretrained_dict.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    for k, v in pretrained_dict.items():
        print(f"Loading params {k} with shape {v.shape}")
        model_dict.update(pretrained_dict)
        model.load_state_dict(model_dict)

model.to(device)

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
gene2idx = vocab.get_stoi()

In [ ]:
# Retrieve the data-independent gene embeddings from scGPT
gene_ids = np.array([id for id in gene2idx.values()])
gene_embeddings = model.encoder(torch.tensor(gene_ids, dtype=torch.long).to(device))
gene_embeddings = gene_embeddings.detach().cpu().numpy()